# CameraTrapML — full experiment on Google Colab

Runs the whole pipeline on a Colab GPU: **download ENA24 → prepare crops → train FP32 baseline →
quantize (QAT + PTQ, int8 … 1bit) → benchmark sweep → figures**.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU` (or better).
2. Push the `colab` branch to GitHub (the notebook clones it).
3. Then `Runtime → Run all`. Expect roughly 2–4 hours on a free T4.

**If Colab disconnects:** every finished stage is copied to Google Drive
(`MyDrive/CameraTrapML_outputs/`). Just reconnect and `Run all` again — finished stages are skipped.

**At the end** download `CameraTrapML_results.zip` (also saved on Drive), unzip it into your local
repo folder, look at `data/figures/`, and commit it yourself.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/VVanlue/CameraTrapML.git"
BRANCH = "colab"

ARCHS = ["mobilenet_v2"]          # add "resnet18" for the second architecture (≈ doubles the run time)
RESOLUTION = 224
BASELINE_EPOCHS = 8
METHODS = ["qat", "ptq"]          # quantization-aware training and post-training quantization
QAT_EPOCHS = 0                    # 0 = per-level defaults (int8 2 ... 1bit 10, 32 total, slowest stage); e.g. 2 for a fast run
BATCH_SIZES = [1, 4, 8, 16]
REPEATS = 3                       # fresh-process repeats per benchmark config
DEVICE_TAG = "colab"              # label for this machine in the results / figures

DRIVE_DIR = "/content/drive/MyDrive/CameraTrapML_outputs"
ENA24_ZIP_URL = "https://storage.googleapis.com/public-datasets-lila/ena24/ena24.zip"    # 3.9 GB, LILA BC
ENA24_JSON_URL = "https://storage.googleapis.com/public-datasets-lila/ena24/ena24.json"

## 2. Check the GPU and mount Google Drive

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## 3. Get the code
If the repo is **private**, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar) with read access to the repo.

In [ ]:
import os, subprocess
REPO = "/content/CameraTrapML"
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret set: fine for a public repo

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Colab already has torch/torchvision/numpy/pandas/sklearn/matplotlib; install the rest.
!pip install -q psutil tqdm pillow

## 4. Restore finished work from Drive
Copies back models / results from a previous (disconnected) session so finished stages are skipped.

In [ ]:
import shutil
from pathlib import Path

SYNC = ["models", "data/results", "data/figures", "data/ena24_boxes.json"]

def save_to_drive():
    for rel in SYNC:
        src, dst = Path(REPO) / rel, Path(DRIVE_DIR) / rel
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif src.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    print("saved to", DRIVE_DIR)

def restore_from_drive():
    for rel in SYNC:
        src, dst = Path(DRIVE_DIR) / rel, Path(REPO) / rel
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif src.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

DONE = Path(DRIVE_DIR) / "done"
DONE.mkdir(parents=True, exist_ok=True)
def done(stage): return (DONE / stage).exists()
def mark(stage):
    save_to_drive()
    (DONE / stage).touch()
    print(f"[stage '{stage}' finished]")

restore_from_drive()
print("finished stages so far:", sorted(p.name for p in DONE.iterdir()) or "none")

## 5. Download ENA24 and prepare the classifier crops (~10–15 min)
Downloaded to the Colab disk (fast, temporary); only the small outputs go to Drive.
`prepare_ena24.py` crops every annotated animal (10% margin) into `data/ena24_crops/<Species>/`.

In [ ]:
from pathlib import Path
RAW = Path("/content/ena24_raw")
CROPS = Path(REPO) / "data/ena24_crops"

if not CROPS.exists() or not any(CROPS.iterdir()):
    RAW.mkdir(parents=True, exist_ok=True)
    if not (RAW / "ena24.json").exists():
        !wget -q -O {RAW}/ena24.json {ENA24_JSON_URL}
    if not (RAW / "ena24.zip").exists():
        !wget -q --show-progress -O {RAW}/ena24.zip {ENA24_ZIP_URL}
    !unzip -q -o {RAW}/ena24.zip -d {RAW}/unzipped
    # the archive's folder layout isn't documented: find the folder that holds the .jpg files
    img_dir = next(p.parent for p in (RAW / "unzipped").rglob("*.jpg"))
    print("images in", img_dir, "-", sum(1 for _ in img_dir.glob("*.jpg")), "files")
    !python run.py prepare --images_dir "{img_dir}" --annotations {RAW}/ena24.json --mode crops
else:
    print("crops already prepared")
print(sum(1 for _ in CROPS.rglob("*.jpg")), "crops in", len(list(CROPS.iterdir())), "classes")

## 6. Train the FP32 baseline (~15–25 min per architecture)

In [ ]:
for arch in ARCHS:
    stage = f"train_{arch}"
    if done(stage):
        print(f"skip {stage} (already done)"); continue
    !python run.py train --arch {arch} --resolution {RESOLUTION} --epochs {BASELINE_EPOCHS} --device cuda --workers 2
    mark(stage)

## 7. Quantize: QAT and PTQ for int8, int4, int3, int2, 1.58bit, 1bit (QAT ≈ 1–2 h per architecture)
QAT walks progressively int8 → int4 → … → 1bit with knowledge distillation from the FP32 model.
PTQ only calibrates (minutes). Each writes `.pt` (integer model), `.qpk` (bit-packed file) and `.json` (report) to `models/`.

In [ ]:
for arch in ARCHS:
    for method in METHODS:
        stage = f"quantize_{arch}_{method}"
        if done(stage):
            print(f"skip {stage} (already done)"); continue
        !python run.py quantize --arch {arch} --resolution {RESOLUTION} --method {method} --fp32_checkpoint models/{arch}_fp32_res{RESOLUTION}.pt --device cuda --workers 2 --epochs {QAT_EPOCHS if method == "qat" else 0}
        mark(stage)

## 8. Benchmark sweep (~30–60 min per architecture)
Accuracy / F1 on the test split, plus latency, throughput and memory on the **Colab CPU**
(the deployment target is a CPU device). Each config runs in a fresh process.

In [ ]:
bs = " ".join(map(str, BATCH_SIZES))
methods = " ".join(METHODS)
for arch in ARCHS:
    stage = f"sweep_{arch}"
    if done(stage):
        print(f"skip {stage} (already done)"); continue
    !python run.py sweep --arch {arch} --resolutions {RESOLUTION} --methods {methods} --batch_sizes {bs} --repeats {REPEATS} --tag {DEVICE_TAG}
    mark(stage)

## 9. Figures and summary table

In [ ]:
for arch in ARCHS:
    !python result.py --arch {arch} --resolution {RESOLUTION} --out_dir data/figures/{arch}
save_to_drive()

from IPython.display import Image, Markdown, display
for arch in ARCHS:
    display(Markdown(f"### {arch}\n" + (Path(REPO) / f"data/figures/{arch}/summary_table.md").read_text()))
    for png in sorted((Path(REPO) / f"data/figures/{arch}").glob("*.png")):
        display(Markdown(f"**{png.name}**")); display(Image(str(png), width=800))

## 10. Download the results
Unzip into your local repo folder (it creates `data/results/`, `data/figures/` and `models/`), check the
figures, then commit. The large quantized models (`models/*.pt`, `.ts`, `.qpk`) are ignored by git;
the zip still contains them so you can run the demo app on any level locally.

In [ ]:
import shutil
zip_base = "/content/CameraTrapML_results"
staging = Path("/content/_results_staging")
shutil.rmtree(staging, ignore_errors=True)
for rel in ["data/results", "data/figures", "models"]:
    shutil.copytree(Path(REPO) / rel, staging / rel, ignore=shutil.ignore_patterns("cache", "*.latent.pt"))
shutil.make_archive(zip_base, "zip", staging)
shutil.copy2(zip_base + ".zip", DRIVE_DIR)
print("saved", zip_base + ".zip", "and a copy in", DRIVE_DIR)

from google.colab import files
files.download(zip_base + ".zip")